# US vs Hong Kong markets: 5 years of growth, risk and co-movement

**Question.** How did US and Hong Kong stock markets compare over the last five years, and how closely do they move together?

**Data.** Daily prices from Yahoo Finance, collected by the DataVest pipeline (`backend/app/pipeline`) into a SQLite database. Metrics come from `backend/app/analytics.py`, which has unit tests.

**What's measured**
- growth (total return and compound annual growth rate, CAGR)
- risk (annualised volatility, maximum drawdown and recovery time)
- return per unit of risk (Sharpe ratio, with a 0% risk-free rate for simplicity)
- co-movement (correlation of returns), including a time-zone effect that changes the answer

*This is a learning project, not investment advice.*

In [ ]:
import sqlite3
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..") / "backend"))
from app import analytics as an
from app.config import WATCHLIST_GROUPS

DB = Path("..") / "backend" / "datavest.db"
NAMES = {s: n for g in WATCHLIST_GROUPS.values() for s, n in g.items()}

plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "font.size": 11})
pd.options.display.float_format = "{:,.3f}".format

In [ ]:
with sqlite3.connect(DB) as con:
    long = pd.read_sql(
        "SELECT t.symbol, p.date, p.adj_close, p.close FROM prices p JOIN tickers t ON t.id = p.ticker_id",
        con, parse_dates=["date"],
    )

long["adj_close"] = long["adj_close"].fillna(long["close"])
prices = long.pivot(index="date", columns="symbol", values="adj_close").sort_index()
prices = prices[[s for s in NAMES if s in prices.columns]]
prices.rename(columns=NAMES).tail()

## 1. Data check

US and Hong Kong exchanges close on different holidays, so the combined table has gaps. The analytics functions handle each market on its own calendar rather than filling gaps with copies of the previous day, which would create fake zero-return days and understate volatility.

In [ ]:
check = pd.DataFrame({
    "first day": prices.apply(lambda s: s.first_valid_index().date()),
    "last day": prices.apply(lambda s: s.last_valid_index().date()),
    "trading days": prices.notna().sum(),
    "days missing vs combined calendar": prices.isna().sum(),
})
check.rename(index=NAMES)

## 2. Growth of $100

Each market is rebased to 100 on its first day. Index levels (S&P 500, Nasdaq, Hang Seng) are **price-only**: they leave out dividends. The ETFs (SPY, 2800.HK) use adjusted prices, which include dividends, so comparing an index with its ETF shows roughly what dividends added.

In [ ]:
core = [s for s in ["^GSPC", "^IXIC", "^HSI", "SPY", "2800.HK"] if s in prices]
rebased = prices[core].apply(lambda s: s.dropna() / s.dropna().iloc[0] * 100)
ax = rebased.ffill().rename(columns=NAMES).plot(title="Growth of $100")
ax.axhline(100, color="grey", lw=1)
ax.set_ylabel("value of $100"); ax.set_xlabel("");

## 3. Return and risk

In [ ]:
summary = an.summary(prices)
table = summary[["total_return", "cagr", "volatility", "sharpe", "max_drawdown"]].copy()
table.index = [NAMES.get(s, s) for s in table.index]
table.sort_values("sharpe", ascending=False).style.format({
    "total_return": "{:+.0%}", "cagr": "{:+.1%}", "volatility": "{:.0%}",
    "sharpe": "{:.2f}", "max_drawdown": "{:.0%}",
}).background_gradient(subset=["sharpe"], cmap="RdYlGn")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
for sym, row in summary.iterrows():
    is_hk = sym.endswith(".HK") or sym == "^HSI"
    ax.scatter(row.volatility, row.cagr, s=70, color="#c2412d" if is_hk else "#0f7b6c")
    ax.annotate(NAMES.get(sym, sym), (row.volatility, row.cagr), xytext=(6, 4), textcoords="offset points", fontsize=9)
ax.axhline(0, color="grey", lw=1)
ax.xaxis.set_major_formatter(lambda x, _: f"{x:.0%}"); ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
ax.set(xlabel="volatility (yearly)", ylabel="growth per year (CAGR)", title="Risk vs return  (red = Hong Kong, green = US)");

## 4. Drawdowns: how deep and how long

A drawdown is the fall from the previous peak. The depth matters, but so does the time it takes to get back to the old high.

In [ ]:
idx = [s for s in ["^GSPC", "^IXIC", "^HSI"] if s in prices]
uw = pd.DataFrame({NAMES[s]: an.drawdown(prices[s]) for s in idx})
ax = uw.plot(title="Falls from the previous peak")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}"); ax.set_xlabel("");

rows = []
for s in prices:
    m = an.max_drawdown(prices[s])
    days = (m["recovered"] - m["trough"]).days if m["recovered"] is not None else None
    rows.append({"market": NAMES[s], "worst fall": m["max_drawdown"], "peak": m["peak"].date(),
                 "bottom": m["trough"].date(), "recovered": m["recovered"].date() if m["recovered"] is not None else "not yet",
                 "days bottom → recovery": days})
pd.DataFrame(rows).set_index("market").sort_values("worst fall").style.format({"worst fall": "{:.0%}"})

## 5. Risk changes over time

Volatility is not constant: it clusters, with calm periods and stormy ones. Below is volatility over rolling 3-month windows.

In [ ]:
rv = an.rolling_volatility(prices[idx]).rename(columns=NAMES)
ax = rv.plot(title="Volatility over rolling 3-month windows")
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}"); ax.set_xlabel("");

## 6. Do Hong Kong and US markets move together?

A naive same-day correlation of daily returns is misleading here. **Hong Kong closes before New York opens**, so news that moves the US on day *t* often reaches Hong Kong on day *t+1*. We compare three measurements:

1. same-day daily returns
2. Hong Kong's return versus the **previous** US day
3. weekly returns, which span both sessions and so avoid the timing problem

In [ ]:
r = an.daily_returns(prices[["^HSI", "^GSPC"]]).dropna()
same_day = r["^HSI"].corr(r["^GSPC"])
us_led = r["^HSI"].corr(r["^GSPC"].shift(1))
weekly = an.correlation(prices[["^HSI", "^GSPC"]], "weekly").loc["^HSI", "^GSPC"]

pd.Series({
    "daily, same day": same_day,
    "daily, HK vs previous US day": us_led,
    "weekly": weekly,
}, name="correlation: Hang Seng vs S&P 500").to_frame()

In [ ]:
corr = an.correlation(prices, "weekly").rename(index=NAMES, columns=NAMES)
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right"); ax.set_yticks(range(len(corr)), corr.index)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(corr.iloc[i, j]) > 0.6 else "black")
ax.grid(False); fig.colorbar(im, shrink=0.8); ax.set_title("Correlation of weekly returns");

In [ ]:
w = an.weekly_returns(prices[["^HSI", "^GSPC"]])
roll = w["^HSI"].rolling(52).corr(w["^GSPC"])
ax = roll.plot(title="Hang Seng vs S&P 500: correlation over rolling 1-year windows (weekly returns)", color="#1b2433")
ax.axhline(weekly, ls="--", color="grey", label=f"5-year average {weekly:.2f}"); ax.legend(); ax.set_xlabel("");

## 7. Findings

The sentences below are generated from the data, so they stay correct every time the notebook is re-run with fresh prices.

In [ ]:
s = summary
def pct(x): return f"{x:+.1%}"
lines = []
if {"^GSPC", "^HSI"} <= set(s.index):
    us, hk = s.loc["^GSPC"], s.loc["^HSI"]
    lines.append(f"Growth: the S&P 500 changed {pct(us.cagr)} a year and the Hang Seng {pct(hk.cagr)} a year (price only, from {us.start.date()}).")
    lines.append(f"Risk: volatility was {us.volatility:.0%} for the S&P 500 and {hk.volatility:.0%} for the Hang Seng; worst falls were {us.max_drawdown:.0%} and {hk.max_drawdown:.0%}.")
    lines.append(f"Co-movement: same-day daily correlation is {same_day:.2f}, but Hong Kong vs the previous US day is {us_led:.2f} and weekly correlation is {weekly:.2f}. "
                 f"{'Measuring across the time-zone gap matters.' if max(us_led, weekly) > same_day + 0.05 else 'The time-zone effect is small in this period.'}")
best = s.sharpe.idxmax(); worst = s.max_drawdown.idxmin()
lines.append(f"Best return per unit of risk: {NAMES[best]} (Sharpe {s.loc[best].sharpe:.2f}). Deepest fall: {NAMES[worst]} ({s.loc[worst].max_drawdown:.0%}).")
print("\n\n".join(lines))

## Limitations and next steps

- **Survivorship and hindsight bias.** The single stocks (e.g. Nvidia) were picked because they are large *today*. Their past results say little about picking stocks in advance.
- **Price-only indexes.** Index levels leave out dividends, which are a meaningful part of Hong Kong returns. The ETFs are fairer comparisons.
- **Currency.** HK assets are priced in HKD, which is pegged to the USD, so currency effects are small here. That would not hold for other markets.
- **Five years is a short window.** Results depend heavily on the start date.
- **Sharpe ratio uses a 0% risk-free rate.** Using actual T-bill or HIBOR rates would lower every Sharpe ratio, by more in recent years when rates were high.

**Next (Phase 3–4):** turn these metrics into API endpoints, then forecast volatility (GARCH against a simple baseline) and cluster stocks by behaviour.